## Math Dataset (Countdown Tasks)


In [ ]:
import os
from datasets import load_dataset

dataset = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", cache_dir="/mnt/lustre/koa/scratch/kantas/hf_home/datasets")

print(f"Train examples: {len(dataset)}")

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/490364 [00:00<?, ? examples/s]

Train examples: 1


## Math Baseline Evaluation (Qwen2.5-Math-1.5B, r1_zero prompt, GSM8k test)

In [ ]:
import json

results_path = "output/Qwen_Math_1.5B_gsm8k_r1zero_results.json"
with open(results_path) as f:
    results = json.load(f)

# Categorize each example
correct     = [r for r in results if r["format_reward"] == 1.0 and r["answer_reward"] == 1.0]
format_only = [r for r in results if r["format_reward"] == 1.0 and r["answer_reward"] == 0.0]
no_format   = [r for r in results if r["format_reward"] == 0.0]

total = len(results)
print(f"Total examples: {total}")
print()
print(f"(1) Correct      (format=1, answer=1): {len(correct):5d}  ({100*len(correct)/total:.1f}%)")
print(f"(2) Format only  (format=1, answer=0): {len(format_only):5d}  ({100*len(format_only)/total:.1f}%)")
print(f"(3) No format    (format=0, answer=0): {len(no_format):5d}  ({100*len(no_format)/total:.1f}%)")

### Cases where format reward = 0 (no `</think> <answer>...</answer>` structure detected)

In [ ]:
import random

sample_no_format = random.sample(no_format, min(10, len(no_format)))

for i, r in enumerate(sample_no_format, 1):
    gen = r["model_generation"]
    has_think_close = "</think>" in gen
    has_answer_open = "<answer>" in gen
    has_answer_close = "</answer>" in gen
    # Check the specific conjunction the reward fn requires
    has_required_junction = "</think> <answer>" in gen

    print(f"--- Example {i} ---")
    print(f"Question: {r['question'][:120]}")
    print(f"Ground truth: {r['ground_truth']}")
    print(f"Has </think>: {has_think_close} | Has <answer>: {has_answer_open} | Has </answer>: {has_answer_close} | Has '</think> <answer>': {has_required_junction}")
    print(f"Generation tail (last 300 chars):\n  {repr(gen[-300:])}")
    print()

**Analysis — format reward = 0:**

The reward function in `drgrpo_grader.py` checks for the *exact* substring `"</think> <answer>"` (with a single space between the two tags). There are two plausible failure modes:

1. **Output truncation** — the model hit `max_new_tokens=2048` before closing its reasoning, so `</think>` was never emitted. Look at the generation tail: if it ends mid-sentence the model simply ran out of budget.

2. **Spacing/format mismatch** — the model emitted `</think>\n<answer>` or `</think><answer>` (no space, or a newline instead of a space). The reward function does *not* use a regex; it requires the literal string `"</think> <answer>"`. If `</think>` is present but the junction is missing, this is a parser strictness issue, not a model failure.

Check the flags above: if `has_think_close=True` but `has_required_junction=False`, the issue is the **parser** (strict space requirement). If `has_think_close=False`, the issue is the **output** (truncation or the model never closed its thinking block).

### Cases where format reward = 1 but answer reward = 0 (wrong answer despite correct structure)

In [ ]:
import re

sample_format_only = random.sample(format_only, min(10, len(format_only)))

for i, r in enumerate(sample_format_only, 1):
    gen = r["model_generation"]
    full_response = "<think>" + gen

    # Extract what the model put in <answer>...</answer>
    answer_match = re.search(r"<answer>(.*?)</answer>", full_response, re.DOTALL)
    extracted_answer = answer_match.group(1).strip() if answer_match else "(none)"

    print(f"--- Example {i} ---")
    print(f"Question: {r['question'][:120]}")
    print(f"Ground truth:      {r['ground_truth']}")
    print(f"Extracted answer:  {extracted_answer}")
    print()

**Analysis — format reward = 1, answer reward = 0:**

These cases formatted correctly but got the math wrong. Common failure patterns to look for:

- **Arithmetic slip** — the reasoning chain is mostly right but makes a calculation error partway through, propagating a wrong final value.
- **Wrong answer extracted** — the model put a `\boxed{}` expression or an intermediate result in `<answer>` instead of the final numeric answer. The grader calls `extract_answer` on anything with `\boxed` inside the answer tag, so a misplaced boxed expression can cause a mismatch.
- **Unit or representation mismatch** — the grader normalizes well but may miss edge cases like `$18.00` vs `18` or a fraction vs decimal if the normalization chain doesn't cover it.
- **Genuinely wrong reasoning** — the 1.5B model simply lacks the arithmetic capability for multi-step word problems without RLVR training.

In [ ]:
import json, glob
points = []
for path in glob.glob("../models/*/training_log.json"):
    log = json.load(open(path))
    meta = next(e for e in log if e.get('meta'))
    acc  = next(e for e in log if e.get('step') == 'final_vllm')
    points.append((meta['num_examples'], acc['val_accuracy']))
points.sort(key=lambda x: x[0] if x[0] != 'full' else 9999)

In [1]:
import os
from datasets import load_dataset


dataset = load_dataset("qwedsacf/competition_math", cache_dir="/mnt/lustre/koa/scratch/kantas/hf_home/datasets")

print(f"Train examples: {len(dataset)}")


Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train examples: 1


In [3]:
from datasets import load_dataset                                                                                                                                                               
import json, os                                                                                                                                                                                 
                                                                                                                                                                                                  
dataset = load_dataset(                                                                                                                                                                         
    "qwedsacf/competition_math",
    cache_dir="/mnt/lustre/koa/scratch/kantas/hf_home/datasets"
)

out_dir = "/mnt/lustre/koa/scratch/kantas/ece405-assignment3-alignment/data/math"                                                                                                               
os.makedirs(out_dir, exist_ok=True)
                                                                                                                                                                                                  
out_path = os.path.join(out_dir, "train.jsonl")                                                                                                                                                 
with open(out_path, "w") as f:
    for item in dataset["train"]:                                                                                                                                                               
        json.dump(item, f)
        f.write("\n")

print(f"Wrote {len(dataset['train'])} examples to {out_path}")

Wrote 12500 examples to /mnt/lustre/koa/scratch/kantas/ece405-assignment3-alignment/data/math/train.jsonl


## Baseline Comparison

Compare grpo_math_12088731.out (no baseline) with grpo_lr_12070555_0.out (reinforced with baseline).

## SFT Curves — Validation Accuracy by Dataset Size

In [ ]:
import json, glob, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MODELS_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "models")

# Each run folder has a training_log.json with entries:
#   {'meta': True, 'num_examples': 'full'|int}
#   {'step': N, 'train_loss': ..., 'val_loss': ..., 'lr': ...}
#   {'step': 'final_vllm', 'val_accuracy': ...}

sft_runs = {}
for log_path in sorted(glob.glob(os.path.join(MODELS_DIR, "*/training_log.json"))):
    folder = os.path.basename(os.path.dirname(log_path))
    # Skip EI and GRPO runs
    if "ei" in folder or "grpo" in folder:
        continue
    log = json.load(open(log_path))
    meta = next((e for e in log if e.get("meta")), None)
    if meta is None:
        continue
    n = meta.get("num_examples", "?")
    steps = [e["step"] for e in log if isinstance(e.get("step"), int)]
    val_losses = [e["val_loss"] for e in log if isinstance(e.get("step"), int) and "val_loss" in e]
    final = next((e.get("val_accuracy") for e in log if e.get("step") == "final_vllm"), None)
    label = f"n={n}" + (f" (acc={final:.1%})" if final is not None else "")
    sft_runs[label] = (steps, val_losses, final)
    print(f"{folder}")
    print(f"  label={label}  steps={len(steps)}  final_acc={final}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, (steps, val_losses, _) in sorted(sft_runs.items()):
    if steps and val_losses:
        axes[0].plot(steps, val_losses, marker="o", markersize=3, linewidth=1.5, label=label)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Validation Loss")
axes[0].set_title("SFT — Val Loss by Dataset Size")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

labels = [l for l, (_, _, acc) in sorted(sft_runs.items()) if acc is not None]
accs   = [acc for _, (_, _, acc) in sorted(sft_runs.items()) if acc is not None]
axes[1].bar(range(len(labels)), [a * 100 for a in accs], color="steelblue", edgecolor="white")
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=15, ha="right", fontsize=8)
axes[1].set_ylabel("Val Accuracy (%)")
axes[1].set_title("SFT — Final Validation Accuracy")
axes[1].axhline(15, color="red", linestyle="--", linewidth=1, label="15% target")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
out = os.path.join(os.path.dirname(os.path.abspath(".")), "latex", "sft_curves.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()

## Expert Iteration Curves — Accuracy & Entropy per EI Step

In [ ]:
import json, glob, os, re
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MODELS_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "models")

# ei_log.json entries: {'ei_step': N, 'val_accuracy': ..., 'entropy': ...,
#                        'n_correct': ..., 'n_rollouts': ...}

ei_runs = {}
for log_path in sorted(glob.glob(os.path.join(MODELS_DIR, "*ei*", "ei_log.json"))):
    folder = os.path.basename(os.path.dirname(log_path))
    # Parse key hyperparams from folder name: G<group>, db<batch>, ep<epochs>
    m = re.search(r"_G(\d+)_db(\d+)_ep(\d+)", folder)
    label = f"G={m.group(1)} db={m.group(2)} ep={m.group(3)}" if m else folder
    log = json.load(open(log_path))
    steps    = [e["ei_step"] for e in log]
    accs     = [e["val_accuracy"] for e in log]
    entropies = [e["entropy"] for e in log]
    ei_runs[label] = (steps, accs, entropies)
    final_acc = accs[-1] if accs else None
    print(f"{label}  →  final_acc={final_acc:.1%}" if final_acc else label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, (steps, accs, entropies) in sorted(ei_runs.items()):
    axes[0].plot(steps, [a * 100 for a in accs], marker="o", linewidth=2, label=label)
    axes[1].plot(steps, entropies, marker="s", linewidth=2, label=label)

axes[0].axhline(15, color="red", linestyle="--", linewidth=1, label="15% target")
axes[0].set_xlabel("EI Step")
axes[0].set_ylabel("Validation Accuracy (%)")
axes[0].set_title("Expert Iteration — Validation Accuracy")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("EI Step")
axes[1].set_ylabel("Mean Token Entropy")
axes[1].set_title("Expert Iteration — Response Entropy")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
out = os.path.join(os.path.dirname(os.path.abspath(".")), "latex", "ei_curves.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()

## GRPO Curves — Reward, Entropy, Grad Norm per Run

In [ ]:
import json, glob, os, re
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MODELS_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "models")
LATEX_DIR  = os.path.join(os.path.dirname(os.path.abspath(".")), "latex")

def grpo_label(folder):
    # Extract loss type and LR if present
    m_lr   = re.search(r"lr([\de\-\+\.]+)$", folder)
    m_loss = re.search(r"grpo_(\w+)_lr", folder)
    loss = m_loss.group(1) if m_loss else "grpo"
    lr   = m_lr.group(1)   if m_lr   else ""
    return f"{loss} lr={lr}" if lr else loss

# grpo_log.json: per-step train metrics
#   keys: grpo_step, lr, train_mean_reward, train_mean_answer_reward,
#         loss, grad_norm, token_entropy
# val_log.json: per-eval val metrics
#   keys: grpo_step, val_mean_reward, val_mean_answer_reward

grpo_runs = {}
for vlog_path in sorted(glob.glob(os.path.join(MODELS_DIR, "*grpo*", "val_log.json"))):
    folder = os.path.basename(os.path.dirname(vlog_path))
    tlog_path = os.path.join(os.path.dirname(vlog_path), "grpo_log.json")
    label = grpo_label(folder)

    val_log   = json.load(open(vlog_path))
    train_log = json.load(open(tlog_path)) if os.path.exists(tlog_path) else []

    val_steps   = [e["grpo_step"]             for e in val_log]
    val_rewards = [e["val_mean_answer_reward"] for e in val_log]

    train_steps   = [e["grpo_step"]               for e in train_log]
    train_rewards = [e["train_mean_answer_reward"] for e in train_log]
    entropies     = [e.get("token_entropy", 0)     for e in train_log]
    grad_norms    = [e.get("grad_norm", 0)         for e in train_log]

    grpo_runs[label] = dict(
        val_steps=val_steps, val_rewards=val_rewards,
        train_steps=train_steps, train_rewards=train_rewards,
        entropies=entropies, grad_norms=grad_norms,
    )
    print(f"{label}  val_steps={len(val_steps)}  final_val_reward={val_rewards[-1]:.4f}" if val_rewards else label)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for label, d in sorted(grpo_runs.items()):
    # Top-left: validation answer reward
    if d["val_steps"]:
        axes[0, 0].plot(d["val_steps"], d["val_rewards"], marker="o", markersize=4,
                        linewidth=2, label=label)
    # Top-right: training answer reward
    if d["train_steps"]:
        axes[0, 1].plot(d["train_steps"], d["train_rewards"], linewidth=1.5, label=label)
    # Bottom-left: token entropy
    if d["train_steps"]:
        axes[1, 0].plot(d["train_steps"], d["entropies"], linewidth=1.5, label=label)
    # Bottom-right: grad norm
    if d["train_steps"]:
        axes[1, 1].plot(d["train_steps"], d["grad_norms"], linewidth=1.5, label=label)

for ax, title, ylabel in [
    (axes[0, 0], "Validation Answer Reward",  "Answer Reward"),
    (axes[0, 1], "Train Answer Reward",        "Answer Reward"),
    (axes[1, 0], "Token Entropy",              "Entropy"),
    (axes[1, 1], "Gradient Norm",              "Grad Norm"),
]:
    ax.set_title(f"GRPO — {title}")
    ax.set_xlabel("GRPO Step")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
out = os.path.join(LATEX_DIR, "grpo_curves.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved: {out}")
plt.show()